# Imports

In [1]:
!pip install datasets lazypredict

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from lightgbm import LGBMRegressor
from lazypredict.Supervised import LazyClassifier
from transformers import AutoModelForSequenceClassification, AutoModel
from huggingface_hub import notebook_login
from typing import Tuple
from modeling_fastesm import FastEsmForMaskedLM

# Dataset Initialization and Creation

In [18]:
# Load your chosen dataset from HuggingFace
dataset_name = "GleghornLab/fitness_prediction"
data = load_dataset(dataset_name)
data

README.md:   0%|          | 0.00/507 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/186k [00:00<?, ?B/s]

valid-00000-of-00001.parquet:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/53.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6289 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/699 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1745 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['seqs', 'labels'],
        num_rows: 6289
    })
    valid: Dataset({
        features: ['seqs', 'labels'],
        num_rows: 699
    })
    test: Dataset({
        features: ['seqs', 'labels'],
        num_rows: 1745
    })
})

In [19]:
df_train = pd.DataFrame(data["train"]).drop_duplicates(subset=['seqs'])
df_test = pd.DataFrame(data["test"]).drop_duplicates(subset=['seqs'])
df_valid = pd.DataFrame(data["valid"]).drop_duplicates(subset=['seqs'])
seq_list_train = df_train['seqs'].tolist()
seq_list_test = df_test['seqs'].tolist()
seq_list_valid = df_valid['seqs'].tolist()

# We extract our labels and sequences from our target dataset
# Also note if you simply want to test a preexisting models embeddings that have already been transformed, you can run the first cell of model training and then skip to the hyperparameter tuning section

In [21]:
# Load model and tokenizer
model_path = 'facebook/esm2_t33_650M_UR50D'
model = FastEsmForMaskedLM.from_pretrained(model_path).eval().cuda()

You are using a model of type esm to instantiate a model of type fast_esm. This is not supported for all configurations of models and can yield errors.


In [ ]:
model_name = "Synthyra/ESMplusplus_small"
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, torch_dtype=torch.float16).cuda()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

modeling_esm_plusplus.py:   0%|          | 0.00/40.7k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Synthyra/ESMplusplus_small:
- modeling_esm_plusplus.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

In [22]:
all_seqs = data['train']['seqs'] + data['valid']['seqs'] + data['test']['seqs']
print(f"Number of sequences: {len(all_seqs)}")
emb_dict = model.embed_dataset(
  sequences=all_seqs, # list of protein strings
  batch_size=8, # embedding batch size
  max_len=20000, # truncate to max_len
  full_embeddings=True, # return residue-wise embeddings
  full_precision=False, # store as float32
  num_workers=10, # data loading num workers
  sql=False, # return dictionary of sequences and embeddings
)

Number of sequences: 8733


Embedding batches:   0%|          | 0/1092 [00:00<?, ?it/s]

In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [24]:
class Pooler:
    def __init__(self, hf_dataset, emb_dict):
        self.train_seqs, self.train_labels = hf_dataset['train']['seqs'], hf_dataset['train']['labels']
        self.valid_seqs, self.valid_labels = hf_dataset['valid']['seqs'], hf_dataset['valid']['labels']
        self.test_seqs, self.test_labels = hf_dataset['test']['seqs'], hf_dataset['test']['labels']
        self.emb_dict = emb_dict

        self.pooling_options = {
            'mean': self.mean_pooling,
            'max': self.max_pooling,
            'min': self.min_pooling,
            'norm': self.norm_pooling,
            'prod': self.prod_pooling,
            'median': self.median_pooling,
            'std': self.std_pooling,
            'var': self.var_pooling,
            'cls': self.cls_pooling,
            'eos': self.eos_pooling
        }

    def mean_pooling(self, emb): # (L, d) -> (d)
        return emb.mean(dim=0)

    def max_pooling(self, emb): # (L, d) -> (d)
        return emb.max(dim=0).values

    def min_pooling(self, emb): # (L, d) -> (d)
        return emb.min(dim=0).values

    def norm_pooling(self, emb): # (L, d) -> (d)
        return emb.norm(dim=0, p=2)

    def prod_pooling(self, emb): # (L, d) -> (d)
        length = emb.shape[0]
        return emb.prod(dim=0) / length

    def median_pooling(self, emb): # (L, d) -> (d)
        return emb.median(dim=0).values

    def std_pooling(self, emb): # (L, d) -> (d)
        return emb.std(dim=0)

    def var_pooling(self, emb): # (L, d) -> (d)
        return emb.var(dim=0)

    def cls_pooling(self, emb): # (L, d) -> (d)
        return emb[0, :]

    def eos_pooling(self, emb): # (L, d) -> (d)
        return emb[-1, :]

    def get_pooled_embeddings(self, pooling_types: Tuple[str, str]): # [mean, max]
        """
        pooler = Pooler(hf_dataset, emb_dict)
        pooling_types = ['mean', 'max']
        X_train, y_train, X_valid, y_valid, X_test, y_test = pooler.get_pooled_embeddings(pooling_types)
        # X (n_samples, n_features)
        # y (n_samples)
        """
        for pool_type in pooling_types:
            if pool_type not in self.pooling_options:
                raise ValueError(f"Invalid pooling type: {pool_type}")

        train_embeddings, valid_embeddings, test_embeddings = [], [], []
        train_labels = np.array(self.train_labels)
        valid_labels = np.array(self.valid_labels)
        test_labels = np.array(self.test_labels)

        # concatenate all pooling for each sequence
        for seq in self.train_seqs:
            final_emb = []
            emb = self.emb_dict[seq] # (L, d)
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb) # (d)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten() # (d * len(pooling_type))
            train_embeddings.append(final_emb)

        train_embeddings = torch.stack(train_embeddings).numpy().astype(np.float32)

        for seq in self.valid_seqs:
            final_emb = []
            emb = self.emb_dict[seq]
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten()
            valid_embeddings.append(final_emb)

        valid_embeddings = torch.stack(valid_embeddings).numpy().astype(np.float32)

        for seq in self.test_seqs:
            final_emb = []
            emb = self.emb_dict[seq]
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten()
            test_embeddings.append(final_emb)

        test_embeddings = torch.stack(test_embeddings).numpy().astype(np.float32)

        return train_embeddings, train_labels, valid_embeddings, valid_labels, test_embeddings, test_labels


In [25]:
pooling_types = ['mean']
pooler = Pooler(data, emb_dict)
X_train, y_train, X_valid, y_valid, X_test, y_test = pooler.get_pooled_embeddings(pooling_types)

In [31]:
def mcc_metric(y_true, y_pred):
    return matthews_corrcoef(y_true, y_pred)

# Set a display name for the function
mcc_metric.__name__ = "mcc_metric"

# Create a sklearn scorer for MCC
mcc_scorer = make_scorer(mcc_metric, greater_is_better=True)

def pearson_metric(y_true, y_pred):
    return pearsonr(y_true, y_pred).correlation

pearson_metric.__name__ = "pearson_metric"
pearson_scorer = make_scorer(pearson_metric, greater_is_better=True)

In [ ]:
split = "username/split_name" # change split name here for testing/tuning
embeddings = load_dataset(f"{split}") # If you already have a dataset with transformed embeddings you can load it here and start tuning hyperparameters and adjusting
X_train = np.array(embeddings['train']['vectors'], dtype='object')
X_valid = np.array(embeddings['validation']['vectors'])
X_test = np.array(embeddings['test']['vectors'], dtype='object')
y_train = np.array(df_train['labels'])
y_valid = np.array(df_valid['labels'])
y_test = np.array(df_test['labels'])

In [15]:
model = RandomForestClassifier() # You can use any other model provided but it must be imported

param_grid = {
    'n_jobs' : [-1],
    'n_estimators': [100, 200, 300, 500],  # Number of trees in the forest
    'max_depth': [5, 10, 20, 30, 50],  # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],  # Minimum number of samples required to be at a leaf node
    'max_features': ['sqrt', 'log2'],  # Number of features to consider when looking for the best split
    'bootstrap': [True, False],  # Whether bootstrap samples are used when building trees
    'criterion': ['gini', 'entropy', 'log_loss'],  # Function to measure the quality of a split
    'class_weight': ['balanced', 'balanced_subsample'],  # Weights associated with classes
}
# Provide a parameter grid to test a wide range of parameters for your chosen model to optimize

grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, cv=5, scoring=mcc_scorer, verbose=4, n_iter = 20) # Creates a grid search instance, note that the scoring method can also be adjusted here

In [38]:
model = RandomForestRegressor() # You can use any other model provided but it must be imported

# Define the parameter grid
param_grid = {
    'n_jobs' : [-1],
    'n_estimators': [100, 200, 300, 500],  # Number of trees in the forest
    'max_depth': [5, 10, 20, 30, 50],  # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],  # Minimum number of samples required to be at a leaf node
    'max_features': ['sqrt', 'log2'],  # Number of features to consider when looking for the best split
    'bootstrap': [True, False],  # Whether bootstrap samples are used when building trees
    'criterion': ['squared_error', 'friedman_mse', 'poisson'],  # Split quality metrics
}
# Provide a parameter grid to test a wide range of parameters for your chosen model to optimize

grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, cv=5, scoring=pearson_scorer, verbose=4, n_iter = 20) # Creates a grid search instance, note that the scoring method can also be adjusted here

In [39]:
grid_search.fit(X_train, y_train) # Fits the parameter grid on a random set of hyperparameters provided in the parameter grid

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END bootstrap=True, criterion=squared_error, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=300, n_jobs=-1;, score=0.496 total time=   6.3s
[CV 2/5] END bootstrap=True, criterion=squared_error, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=300, n_jobs=-1;, score=0.616 total time=   5.7s
[CV 3/5] END bootstrap=True, criterion=squared_error, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=300, n_jobs=-1;, score=0.738 total time=   5.6s
[CV 4/5] END bootstrap=True, criterion=squared_error, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=300, n_jobs=-1;, score=0.791 total time=   5.3s
[CV 5/5] END bootstrap=True, criterion=squared_error, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=300, n_jobs=-1;, score=0.745 total time= 

RandomizedSearchCV(cv=5, estimator=RandomForestRegressor(), n_iter=20,
                   param_distributions={'bootstrap': [True, False],
                                        'criterion': ['squared_error',
                                                      'friedman_mse',
                                                      'poisson'],
                                        'max_depth': [5, 10, 20, 30, 50],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500],
                                        'n_jobs': [-1]},
                   scoring=make_scorer(pearson_metric, response_method='predict'),
                   verbose=4)

In [40]:
best_params = grid_search.best_params_
best_estimator = grid_search.best_estimator_
best_score = grid_search.best_score_

print(best_params)
print(best_estimator) # Shows the best hyperparameters for the model

{'n_jobs': -1, 'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 20, 'criterion': 'poisson', 'bootstrap': False}
RandomForestRegressor(bootstrap=False, criterion='poisson', max_depth=20,
                      max_features='sqrt', min_samples_leaf=4,
                      min_samples_split=10, n_jobs=-1)
